In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import pycountry
from scipy import stats
from scipy.signal import savgol_filter

# Cargar datos de proyeccion PIB
Proyeccion_PIB = '../../data/fuentes/economicos/Proyeccion_PIB_indicepais.xlsx'
df_ImpactoEconomico = pd.read_excel(Proyeccion_PIB, sheet_name='Indice de impacto económico')


#Trasponer columnas de escenarios

df_ImpactoEconomico2=df_ImpactoEconomico.melt(
    id_vars = ["País"],
    value_vars= [
        "Riesgo crónico (Impacto en PIB)",
        "Score de riesgo de sequía",
        "Score de riesgo de clima lluvioso",
        "Capacidad actual de adaptación",
        "Índice de impacto económico"
    ],

    var_name= "Indicador",
    value_name= "Valor"
)

# Transformar en formato número

df_ImpactoEconomico2["Valor"] = (
        df_ImpactoEconomico2["Valor"].astype(str)
        .str.replace("–", "-", regex=False)           
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
)

df_ImpactoEconomico2["Valor"]=pd.to_numeric(df_ImpactoEconomico2["Valor"], errors="coerce")

print(df_ImpactoEconomico2)

            País                        Indicador  Valor
0        Finland  Riesgo crónico (Impacto en PIB)    3.0
1    Switzerland  Riesgo crónico (Impacto en PIB)    4.0
2        Austria  Riesgo crónico (Impacto en PIB)    7.0
3       Portugal  Riesgo crónico (Impacto en PIB)    9.0
4         Canada  Riesgo crónico (Impacto en PIB)   12.0
..           ...                              ...    ...
235     Thailand      Índice de impacto económico   36.0
236        India      Índice de impacto económico   36.4
237  Philippines      Índice de impacto económico   37.3
238     Malaysia      Índice de impacto económico   38.3
239    Indonesia      Índice de impacto económico   39.2

[240 rows x 3 columns]


In [2]:
import pandas as pd
import numpy as np
import pymysql
from pymysql.constants import CLIENT
from dotenv import load_dotenv
import os

load_dotenv()

# Obtener los parámetros de conexión
DB_HOST = os.getenv('DB_HOST')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_NAME = os.getenv('DB_NAME')

conexion = pymysql.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    client_flag=CLIENT.MULTI_STATEMENTS
)
cursor = conexion.cursor()

# 2) Cargar dimensión Paises: (codigo, nombre_en) → dict nombre_en_normalizado → codigo
cursor.execute("SELECT codigo, nombre_en FROM Paises;")
dim_paises = {
    nombre_en.strip().lower(): codigo
    for codigo, nombre_en in cursor.fetchall()
}


# 1) Normalizar columna "País" → minúsculas y sin espacios
df_ImpactoEconomico2["pais_norm"] = (
    df_ImpactoEconomico2["País"]
      .astype(str)
      .str.strip()
      .str.lower()
)

# 2) Diccionario de excepciones (adaptado a tus datos)
exceptions = {
    'netherlands':'netherlands (kingdom of the)',
    'turkiye':'türkiye',
    'united kingdom':'united kingdom of great britain and northern ireland',
    'bahamas, the':'bahamas',
    'bolivia':'bolivia (plurinational state of)',
    'congo, dem. rep.':'congo (the democratic republic of the)',
    'congo, rep.':'congo',
    "cote d'ivoire":"côte d'ivoire",
    'egypt, arab rep.':'egypt',
    'gambia, the':'gambia',
    'hong kong sar, china':'hong kong',
    'iran, islamic rep.':'iran (islamic republic of)',
    'korea, rep.':'korea (the republic of)',
    'micronesia, fed. states of':'micronesia (federated states of)',
    'st. vincent and the grenadines':'saint vincent and the grenadines',
    'tanzania':'tanzania, the united republic of',
    'curacao':'curaçao',
    "korea, dem. people's rep.":"korea (the democratic people's republic of)",
    'slovak republic':'slovakia',
    'venezuela, rb':'venezuela (bolivarian republic of)',
    'yemen, rep.':'yemen',
    'st. kitts and nevis':'saint kitts and nevis',
    'st. lucia':'saint lucia',
    'macao sar, china':'macao',
    'lao pdr':"lao people's democratic republic",
    'kyrgyz republic':'kyrgyzstan',
    'russian federation':'russian federation',
    'moldova':'moldova (the republic of)',
    'united states':'united states of america',
    'us':'united states of america',
    'st. martin (french part)':'saint martin (french part)',
    'british virgin islands':'virgin islands (british)',
    'venezuela':'Venezuela (Bolivarian Republic of)',
    'korea':'Korea (the Republic of)',
    'czech': 'Czechia',
    'russia': 'Russian Federation',
    'taiwan': 'Taiwan (Province of China)',
    'turkey': 'Türkiye',
    'uae': 'United Arab Emirates',
    'uk': 'United Kingdom of Great Britain and Northern Ireland',   

}

# 3) Aplicar excepciones → columna intermedia
df_ImpactoEconomico2["pais_db"] = df_ImpactoEconomico2["pais_norm"].map(
    lambda x: exceptions[x] if x in exceptions else x
)

# 4) Mapear con la dimensión de países cargada previamente
# (asegúrate de tener dim_paises = {nombre_en.lower(): codigo})
df_ImpactoEconomico2["pais_id"] = (
    df_ImpactoEconomico2["pais_db"]
      .str.strip()
      .str.lower()
      .map(dim_paises)
)

# 5) Listar países no mapeados
no_map = df_ImpactoEconomico2.loc[df_ImpactoEconomico2["pais_id"].isna(), "País"].unique()
print("⚠️ Países sin mapeo:", no_map)

df_ImpactoEconomico2

⚠️ Países sin mapeo: []


,País,Indicador,Valor,pais_norm,pais_db,pais_id
0,Finland,Riesgo crónico (Impacto en PIB),3.0,finland,finland,FI
1,Switzerland,Riesgo crónico (Impacto en PIB),4.0,switzerland,switzerland,CH
2,Austria,Riesgo crónico (Impacto en PIB),7.0,austria,austria,AT
3,Portugal,Riesgo crónico (Impacto en PIB),9.0,portugal,portugal,PT
4,Canada,Riesgo crónico (Impacto en PIB),12.0,canada,canada,CA
...,...,...,...,...,...,...
235,Thailand,Índice de impacto económico,36.0,thailand,thailand,TH
236,India,Índice de impacto económico,36.4,india,india,IN
237,Philippines,Índice de impacto económico,37.3,philippines,philippines,PH
238,Malaysia,Índice de impacto económico,38.3,malaysia,malaysia,MY


In [3]:
# Diccionario de mapeo: Indicador → id en tabla Indicadores
indicadores_map = {
    "Riesgo crónico (Impacto en PIB)": 39,
    "Score de riesgo de sequía": 40,
    "Score de riesgo de clima lluvioso": 41,
    "Capacidad actual de adaptación": 42,
    "Índice de impacto económico": 43
}

# Mapear desde la columna "Indicador" a indicador_id
df_ImpactoEconomico2["indicador_id"] = df_ImpactoEconomico2["Indicador"].map(indicadores_map)

# Agregar columna de año (ejemplo: todos 2024 o el que corresponda)
df_ImpactoEconomico2["anio"] = 2020

# Filtrar solo registros con mapeo válido y país mapeado
df_hechos = df_ImpactoEconomico2.loc[
    df_ImpactoEconomico2["pais_id"].notna() & df_ImpactoEconomico2["indicador_id"].notna(),
    ["anio", "Valor", "pais_id", "indicador_id"]
].rename(columns={"Valor": "valor"}).copy()


print(df_hechos.head(20))


    anio  valor pais_id  indicador_id
0   2020    3.0      FI            39
1   2020    4.0      CH            39
2   2020    7.0      AT            39
3   2020    9.0      PT            39
4   2020   12.0      CA            39
5   2020    6.0      NO            39
6   2020   13.0      US            39
7   2020   10.0      SE            39
8   2020    1.0      DK            39
9   2020   17.0      DE            39
10  2020   22.0      JP            39
11  2020   14.0      ES            39
12  2020   28.0      GR            39
13  2020   33.0      AU            39
14  2020   11.0      GB            39
15  2020   15.0      TR            39
16  2020    5.0      NL            39
17  2020   29.0      NZ            39
18  2020   31.0      IT            39
19  2020   24.0      KR            39


In [4]:
sql_insert = """
INSERT INTO Hechos (anio, valor, pais_id, indicador_id)
VALUES (%s, %s, %s, %s)
"""

cursor.executemany(sql_insert, df_hechos.values.tolist())
conexion.commit()
print(f"✅ Se insertaron {cursor.rowcount} registros en la tabla Hechos")

✅ Se insertaron 240 registros en la tabla Hechos
